<a href="https://colab.research.google.com/github/hasmalee/aeropinn/blob/main/AeroPINN_X_Exact20_Stage1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AeroPINN_X_Exact20_Stage1

This notebook is a **starter exact-20 milestone notebook** for safer training:
- 20 different airfoils
- 1 fixed `(AoA, Re)` case per airfoil
- 20 total blocks
- checkpoint after every block, plus stronger validation after each block

Use the bundled `train_loop.py` and `postprocess.py` from this folder.
This notebook is a guided template derived from your existing training notebook, not a mathematical guarantee of final aerodynamic accuracy.

In [32]:
# ============================================================================
# CELL 1 — Imports & Environment
# ============================================================================
from __future__ import annotations
import os, gc, json, math, random
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

from airfoil import AirfoilGeometry, AirfoilFormatError
from parsec_fit import (
    fit_parsec_to_dat, fit_quality_report,
    preprocess_airfoil_for_parsec, parsec_array_to_dict,
)
from network import MLP, load_checkpoint, save_checkpoint
from train_loop import train_multi_airfoil, optimize_shape
from postprocess import (
    compute_lift_drag, validate_model_on_library,
    export_optimized_airfoil, export_run_report,
)
print("All imports OK")

Device: cuda
GPU: NVIDIA A100-SXM4-40GB
All imports OK


In [33]:
# ============================================================================
# CELL 2 — Folder Paths  (edit OUT_DIR and DATA_ROOT to match your Drive)
# ============================================================================
DATA_ROOT = Path("/content/airfoils")
TRAIN_DIR = DATA_ROOT / "train"
VAL_DIR   = DATA_ROOT / "val"

OUT_DIR   = Path("/content/drive/MyDrive/AeroPINN")
CKPT_DIR  = OUT_DIR / "checkpoints"
VAL_OUT   = OUT_DIR / "validation"

for d in [CKPT_DIR, VAL_OUT]:
    d.mkdir(parents=True, exist_ok=True)

# Set to None to start fresh, or supply a path to resume
RESUME_CHECKPOINT = None   # e.g. OUT_DIR / "multi_airfoil_last.pt"
print("Folders OK")

Folders OK


In [34]:
# ============================================================================
# CELL 3 — Training hyper-parameters
# ============================================================================
#
# TARGET: 20 combinations (5 airfoils × 2 AoA × 2 Re).
# Each combo gets BLOCK_STEPS Adam steps.  After every 10 blocks a
# checkpoint is saved, so 20 combos = 2 checkpoint saves.
#
# Physics range that matches the HTML reference table:
#   AoA  : 0°, 4°
#   Re   : 1e6 (1M), 2e6 (2M)
# ─────────────────────────────────────────────────────────────────────────────
AOA_LIST = [0.0, 4.0]          # degrees — matches ref table
RE_LIST  = [1.0e6, 2.0e6]      # 1M and 2M — in the 10^6‒20^6 band you asked

XLIM = (-1.0, 2.0)
YLIM = (-1.0, 1.0)
NEAR_BAND = 0.003

# Point-cloud sizes (research-grade, still GPU-safe)
N_INT     = 3000
N_NEAR    = 600
N_WAKE    = 600
N_AIRFOIL = 800
N_INLET   = 500
N_OUTLET  = 500
N_TOP     = 500
N_BOT     = 500

# Optimiser
LR_ADAM      = 1e-4
LR_MIN       = 2e-5
LR_LAMBDA    = 3e-4
WEIGHT_DECAY = 1e-5

# Adaptive sampler
ADAPTIVE_EVERY       = 300
ADAPTIVE_CAP         = 2000
CANDIDATE_POOL       = 6000
GAUSS_PER_CENTER     = 6
RESIDUAL_PROBE_STRIDE = 4

# Logging
PRINT_EVERY = 100
SAVE_EVERY  = 500
VAL_EVERY   = 2

# Block schedule
BLOCK_STEPS     = 1500   # Adam steps per airfoil×condition block
BLOCKS_PER_RUN  = 10    # checkpoint every 10 blocks
MAX_TOTAL_COMBOS = 20   # 5 airfoils × 2 AoA × 2 Re

LBFGS_STEPS = 10        # final polish after all 20 combos

PARSEC_OPT_THRESHOLD = 4.5

print(f"AOA_LIST          = {AOA_LIST}")
print(f"RE_LIST           = {RE_LIST}")
print(f"BLOCK_STEPS       = {BLOCK_STEPS}")
print(f"MAX_TOTAL_COMBOS  = {MAX_TOTAL_COMBOS}")
print(f"Total Adam steps  = {BLOCK_STEPS * MAX_TOTAL_COMBOS:,}")

AOA_LIST          = [0.0, 4.0]
RE_LIST           = [1000000.0, 2000000.0]
BLOCK_STEPS       = 1500
MAX_TOTAL_COMBOS  = 20
Total Adam steps  = 30,000


In [35]:
# ============================================================================
# CELL 4 — Build airfoil library from .dat files
# ============================================================================
# These 20 airfoils span the reference table and include all 20 unique names
# that appear in the HTML: eppler378, mh60, mh78, naca0009, naca0012-64,
# naca0015, naca1410, naca23012, naca2412, naca4412, naca4415,
# naca63_012, naca6412, naca64_012, naca65_210, naca66_212,
# s1223, s809, s814, sd7034
# You need these .dat files under /content/airfoils/train/
# ─────────────────────────────────────────────────────────────────────────────

TRAINING_AIRFOIL_NAMES = [
    # cambered / high-lift family ─ gives large positive CL at 4°
    "naca2412",   # Ref: CL=0.6510, CD=0.0072, L/D=90.40
    "naca4412",   # Ref: CL=0.7842, CD=0.0061, L/D=128.10 (4°/1M)
    "naca4415",   # Ref: CL=0.7850, CD=0.0066, L/D=118.90 (4°/2M)
    "naca6412",   # Ref: CL=0.6320, CD=0.0087, L/D=72.60  (4°/1M)
    "naca65_210", # Ref: CL=0.5620, CD=0.0072, L/D=78.10  (4°/2M)
    "naca66_212", # Ref: CL=0.6270, CD=0.0063, L/D=99.50  (4°/2M)
    "naca23012",  # Ref: CL=0.4280, CD=0.0058, L/D=73.80  (4°/2M)
    "naca23015",  # Ref: CL=0.4310, CD=0.0064, L/D=67.30  (4°/2M)
    "naca1410",   # Ref: CL=0.4212, CD=0.0064, L/D=66.00  (4°/1M)
    # symmetric — CL≈0 at 0° but positive at 4°
    "naca0009",   # Ref: CL=0.4250, CD=0.0062, L/D=68.50  (4°/2M)
    "naca0012-64",# Ref: CL=0.4330, CD=0.0066, L/D=66.10  (4°/2M)
    "naca0015",   # Ref: CL=0.4352, CD=0.0084, L/D=51.80  (4°/1M)
    "naca0018",   # Ref: CL=0.4420, CD=0.0072, L/D=61.40  (4°/2M)
    "naca63_012", # Ref: CL=0.4426, CD=0.0107, L/D=41.30  (4°/1M)
    "naca64_012", # Ref: CL=0.4340, CD=0.0064, L/D=67.80  (4°/2M)
    # high-performance / special shapes
    "s1223",      # Ref: CL=0.6490, CD=0.0082, L/D=79.10  (4°/1M)
    "sd7034",     # Ref: CL=0.6750, CD=0.0078, L/D=86.50  (4°/1M)
    "eppler378",  # Ref: CL=1.0650, CD=0.0082, L/D=129.90 (4°/1M)
    "mh60",       # Ref: CL=0.6110, CD=0.0068, L/D=89.90  (4°/1M)
    "mh78",       # Ref: CL=0.6890, CD=0.0071, L/D=97.00  (4°/2M)
]

# Validation set — small held-out group
VAL_AIRFOIL_NAMES = ["naca0009", "naca2412", "naca4412", "s1223"]


def load_airfoil_library(folder: Path, names=None, n_boundary: int = 1200):
    """Load airfoils from .dat files in folder."""
    lib, skipped = [], []
    files = sorted(folder.glob("*.dat"))
    if names is not None:
        files = [f for f in files if f.stem in names]

    for f in files:
        try:
            geom = AirfoilGeometry.from_dat(f, n_boundary=n_boundary)
            parsec_params = None
            fit_report    = None
            fit_mse       = None
            use_for_opt   = False
            parsec_quality = "POOR"

            try:
                pre = preprocess_airfoil_for_parsec(geom.boundary, n_surface=300)
                params, fit_mse = fit_parsec_to_dat(str(f), verbose=False)
                fit_report = fit_quality_report(pre["upper"], pre["lower"], params)
                if fit_report["max_error_pct"] <= PARSEC_OPT_THRESHOLD:
                    parsec_params  = params
                    use_for_opt    = True
                    parsec_quality = fit_report["quality"]
            except Exception as e_fit:
                fit_report = {
                    "quality": "POOR", "max_error_pct": 100.0,
                    "mean_error_pct": 100.0, "error_message": str(e_fit),
                }

            lib.append({
                "name":                f.stem,
                "path":                str(f),
                "geometry":            geom,
                "parsec_params":       parsec_params,
                "fit_report":          fit_report,
                "fit_mse":             fit_mse,
                "use_for_optimization": use_for_opt,
                "parsec_quality":      parsec_quality,
            })
        except Exception as e:
            skipped.append((f.name, str(e)))

    return lib, skipped


TRAIN_AIRFOILS, TRAIN_SKIP = load_airfoil_library(TRAIN_DIR, TRAINING_AIRFOIL_NAMES)
VAL_AIRFOILS,   VAL_SKIP   = load_airfoil_library(VAL_DIR,   VAL_AIRFOIL_NAMES)

print(f"Loaded {len(TRAIN_AIRFOILS)} training airfoils")
print(f"Loaded {len(VAL_AIRFOILS)}   validation airfoils")
if TRAIN_SKIP or VAL_SKIP:
    print("Skipped:", TRAIN_SKIP + VAL_SKIP)

Loaded 20 training airfoils
Loaded 0   validation airfoils


In [51]:
# ============================================================================
# CELL — Load exact 20-case schedule
# ============================================================================
import json
from pathlib import Path

SCHEDULE_PATH = Path("exact_20_combo_schedule.json")
EXACT_20_SCHEDULE = json.loads(SCHEDULE_PATH.read_text())

print(f"Loaded {len(EXACT_20_SCHEDULE)} fixed cases")
for row in EXACT_20_SCHEDULE[:5]:
    print(row)

Loaded 20 fixed cases
{'block_id': 1, 'airfoil': 'naca2412', 'alpha_deg': 0.0, 'Re_phys': 1000000.0, 'note': 'starter exact-20 schedule'}
{'block_id': 2, 'airfoil': 'naca4412', 'alpha_deg': 0.0, 'Re_phys': 2000000.0, 'note': 'starter exact-20 schedule'}
{'block_id': 3, 'airfoil': 'naca4415', 'alpha_deg': 2.0, 'Re_phys': 1000000.0, 'note': 'starter exact-20 schedule'}
{'block_id': 4, 'airfoil': 'naca6412', 'alpha_deg': 2.0, 'Re_phys': 2000000.0, 'note': 'starter exact-20 schedule'}
{'block_id': 5, 'airfoil': 'naca65_210', 'alpha_deg': 4.0, 'Re_phys': 1000000.0, 'note': 'starter exact-20 schedule'}


In [52]:
# ============================================================================
# CELL 5 — Build or resume model
# ============================================================================
model = MLP(
    in_dim=5,
    out_dim=4,
    hidden_dim=64,
    num_hidden_layers=16,
    activation="silu",
).to(device)

if RESUME_CHECKPOINT is not None and Path(RESUME_CHECKPOINT).exists():
    ckpt = torch.load(RESUME_CHECKPOINT, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state"])
    print("Resumed from:", RESUME_CHECKPOINT)

print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

MLP(5→4, 64×16, act=silu, params=64,068)
Parameters: 64,068


In [55]:
import importlib

import airfoil
import losses
import network
import parsec_fit
import residuals
import sampling
import postprocess
import train_loop
import xai

importlib.reload(airfoil)
importlib.reload(losses)
importlib.reload(network)
importlib.reload(parsec_fit)
importlib.reload(residuals)
importlib.reload(sampling)
importlib.reload(postprocess)
importlib.reload(train_loop)
importlib.reload(xai)

print("Reloaded:")
print("airfoil     ->", airfoil.__file__)
print("losses      ->", losses.__file__)
print("network     ->", network.__file__)
print("parsec_fit  ->", parsec_fit.__file__)
print("residuals   ->", residuals.__file__)
print("sampling    ->", sampling.__file__)
print("postprocess ->", postprocess.__file__)
print("train_loop  ->", train_loop.__file__)
print("xai         ->", xai.__file__)

Reloaded:
airfoil     -> /content/airfoil.py
losses      -> /content/losses.py
network     -> /content/network.py
parsec_fit  -> /content/parsec_fit.py
residuals   -> /content/residuals.py
sampling    -> /content/sampling.py
postprocess -> /content/postprocess.py
train_loop  -> /content/train_loop.py
xai         -> /content/xai.py


In [56]:
import inspect
print(inspect.signature(train_loop.train_multi_airfoil))

(model, train_airfoils: 'Sequence[dict[str, Any]]', val_airfoils: 'Sequence[dict[str, Any]]', aoa_list: 'Sequence[float]', re_list: 'Sequence[float]', xlim, ylim, N_int: 'int', N_near: 'int', N_airfoil: 'int', N_inlet: 'int', N_outlet: 'int', N_top: 'int', N_bot: 'int', N_wake: 'int', near_band: 'float', adam_steps: 'int', lr_adam: 'float', lr_min: 'float', lr_lambda: 'float', weight_decay: 'float', refresh_every: 'int', adaptive_every: 'int', adaptive_cap: 'int', candidate_pool: 'int', gauss_per_center: 'int', residual_probe_stride: 'int', lbfgs_steps: 'int', print_every: 'int', save_every: 'int', val_every: 'int', out_dir: 'str', device: 'str', seed: 'int' = 42, block_steps: 'int' = 500, pde_chunk_size: 'int' = 1500, score_chunk_size: 'int' = 2000, resume_checkpoint: 'str | None' = None, blocks_per_run: 'int' = 10, max_total_combos: 'int' = 100)


In [57]:
from residuals import pde_residuals_rans_sa
from postprocess import compute_lift_drag_on_boundary, compute_LD_differentiable
from sampling import build_point_cloud, cloud_to_tensors
from losses import total_loss, AdaptiveLossWeights
from train_loop import train_multi_airfoil

In [59]:
# ============================================================================
# SAFE RETRY — train only block 1 again with explicit 1500 steps
# ============================================================================

import torch
import train_loop

device = "cuda" if torch.cuda.is_available() else "cpu"

SAFE_OUT = "/content/drive/MyDrive/AeroPINN/safe_block1_retry"

ONE_AIRFOIL = [next(a for a in TRAIN_AIRFOILS if a["name"] == "naca2412")]
ONE_AOA = [0.0]
ONE_RE = [1.0e6]

model, adaptive_weights, history = train_loop.train_multi_airfoil(
    model=model,
    train_airfoils=ONE_AIRFOIL,
    val_airfoils=[],
    aoa_list=ONE_AOA,
    re_list=ONE_RE,
    xlim=XLIM,
    ylim=YLIM,
    N_int=N_INT,
    N_near=N_NEAR,
    N_airfoil=N_AIRFOIL,
    N_inlet=N_INLET,
    N_outlet=N_OUTLET,
    N_top=N_TOP,
    N_bot=N_BOT,
    N_wake=N_WAKE,
    near_band=NEAR_BAND,

    # keep these explicit
    adam_steps=BLOCK_STEPS,
    block_steps=BLOCK_STEPS,

    lr_adam=LR_ADAM,
    lr_min=LR_MIN,
    lr_lambda=LR_LAMBDA,
    weight_decay=WEIGHT_DECAY,

    refresh_every=ADAPTIVE_EVERY,
    adaptive_every=ADAPTIVE_EVERY,
    adaptive_cap=ADAPTIVE_CAP,
    candidate_pool=CANDIDATE_POOL,
    gauss_per_center=GAUSS_PER_CENTER,
    residual_probe_stride=RESIDUAL_PROBE_STRIDE,

    lbfgs_steps=0,          # keep 0 for now during diagnosis
    print_every=PRINT_EVERY,
    save_every=SAVE_EVERY,
    val_every=VAL_EVERY,

    out_dir=SAFE_OUT,
    device=device,
    seed=42,
    blocks_per_run=1,
    max_total_combos=1,
    pde_chunk_size=1500,
    score_chunk_size=2000,
    resume_checkpoint=None,
)

train_multi_airfoil: 1 airfoils, 1 conditions, selected combos=1, running blocks 0..0

BLOCK 1/1
  Airfoil : naca2412
  AoA     : 0.0 deg
  Re      : 1.000e+06
  Steps   : 1500
  step   100  loss=2.0445e-01  pde=1.2446e-02  wall=5.8394e-02  lr=8.98e-05  λwall=0.43  λpde=0.13
  step   200  loss=1.2131e-02  pde=3.2388e-02  wall=2.6679e-04  lr=8.07e-05  λwall=0.17  λpde=0.22
  step   300  loss=1.6050e-02  pde=2.2157e-02  wall=2.6486e-04  lr=7.25e-05  λwall=0.13  λpde=0.66
  step   400  loss=1.9130e-02  pde=1.1137e-02  wall=2.0177e-04  lr=6.51e-05  λwall=0.13  λpde=1.38
  step   500  loss=9.5177e-03  pde=3.5263e-03  wall=2.7642e-04  lr=5.85e-05  λwall=0.13  λpde=2.08
  step   600  loss=3.6119e-03  pde=1.2928e-03  wall=7.7316e-05  lr=5.25e-05  λwall=0.13  λpde=2.13
  step   700  loss=2.0633e-03  pde=6.6381e-04  wall=2.9169e-05  lr=4.72e-05  λwall=0.13  λpde=2.13
  step   800  loss=1.5094e-03  pde=4.6288e-04  wall=1.6033e-05  lr=4.24e-05  λwall=0.13  λpde=2.13
  step   900  loss=1.2166e-03  

In [61]:
data = cp_on_boundary(
    model=model,
    geometry=airfoil_item["geometry"],
    alpha_deg=0.0,
    Re_phys=1e6,
    n_surface=240,
    xlim=(-1.0, 2.0),
    ylim=(-1.0, 1.0),
    device=device,
)

print("p_ref =", data["p_ref"])
print("x[:5]  =", data["x"][:5])
print("y[:5]  =", data["y"][:5])
print("p[:5]  =", data["p"][:5])
print("cp[:5] =", data["cp"][:5])

NameError: name 'cp_on_boundary' is not defined

In [62]:
from pathlib import Path
import json
import torch
from postprocess import compute_lift_drag_on_boundary

device = "cuda" if torch.cuda.is_available() else "cpu"
RUN_DIR = Path("/content/drive/MyDrive/AeroPINN/safe_block1_retry")

schedule = json.loads((RUN_DIR / "combo_schedule.json").read_text())
case0 = schedule[0]
print("Case 0:", case0)

airfoil_name = case0["airfoil"]
alpha_deg = float(case0["alpha_deg"])
Re_phys = float(case0["Re_phys"])

airfoil_item = next(a for a in TRAIN_AIRFOILS if a["name"] == airfoil_name)

metrics = compute_lift_drag_on_boundary(
    model=model,
    geometry=airfoil_item["geometry"],
    alpha_deg=alpha_deg,
    Re_phys=Re_phys,
    n_surface=240,
    xlim=(-1.0, 2.0),
    ylim=(-1.0, 1.0),
    device=device,
)

print("\nBLOCK-1 METRICS")
print("CL  =", float(metrics["CL"]))
print("CD  =", float(metrics["CD"]))
print("L/D =", float(metrics["LD"]))

Case 0: {'airfoil': 'naca2412', 'alpha_deg': 0.0, 'Re_phys': 1000000.0}

BLOCK-1 METRICS
CL  = 0.00017630283291225624
CD  = 4.6078744750488454e-05
L/D = 3.8261205652827024


old

In [29]:
# ============================================================================
# CELL — Safer exact-20 training loop (1 block at a time)
# ============================================================================
from pathlib import Path
import json, torch
from postprocess import compute_lift_drag_on_boundary

SAFE_OUT = OUT_DIR / "safe_exact20"
SAFE_OUT.mkdir(parents=True, exist_ok=True)

history_rows = []
for row in EXACT_20_SCHEDULE:
    af_name = row["airfoil"]
    alpha_deg = float(row["alpha_deg"])
    Re_phys = float(row["Re_phys"])

    target_items = [a for a in TRAIN_AIRFOILS if a["name"] == af_name]
    if not target_items:
        print(f"SKIP missing airfoil: {af_name}")
        continue
    target = target_items[0]

    print("=" * 72)
    print(f"BLOCK {row['block_id']:02d}/20 | {af_name} | AoA={alpha_deg} | Re={Re_phys:.0e}")

    model, adaptive_weights, history = train_multi_airfoil(
        model=model,
        train_airfoils=[target],
        val_airfoils=VAL_AIRFOILS,
        aoa_list=[alpha_deg],
        re_list=[Re_phys],
        xlim=XLIM,
        ylim=YLIM,
        N_int=N_INT,
        N_near=N_NEAR,
        N_airfoil=N_AIRFOIL,
        N_inlet=N_INLET,
        N_outlet=N_OUTLET,
        N_top=N_TOP,
        N_bot=N_BOT,
        N_wake=N_WAKE,
        near_band=NEAR_BAND,
        adam_steps=BLOCK_STEPS,
        lr_adam=LR_ADAM,
        lr_min=LR_MIN,
        lr_lambda=LR_LAMBDA,
        weight_decay=WEIGHT_DECAY,
        refresh_every=ADAPTIVE_EVERY,
        adaptive_every=ADAPTIVE_EVERY,
        adaptive_cap=ADAPTIVE_CAP,
        candidate_pool=CANDIDATE_POOL,
        gauss_per_center=GAUSS_PER_CENTER,
        residual_probe_stride=RESIDUAL_PROBE_STRIDE,
        print_every=PRINT_EVERY,
        save_every=SAVE_EVERY,
        val_every=VAL_EVERY,
        lbfgs_steps=0,
        out_dir=SAFE_OUT,
        device=device,
    )

    metrics = compute_lift_drag_on_boundary(
    model=model,
    geometry=geometry,
    alpha_deg=alpha_deg,
    Re_phys=Re_phys,
    n_surface=240,
    xlim=xlim,
    ylim=ylim,
    device=device,
    )

    CL = float(metrics["CL"])
    CD = float(metrics["CD"])
    LD = float(metrics["LD"])

    ckpt_path = SAFE_OUT / f"ckpt_block_{row['block_id']:02d}.pt"
    torch.save({"model_state": model.state_dict(),
                "adaptive_weights": adaptive_weights,
                "last_case": row,
                "metrics": metrics}, ckpt_path)

    out = dict(row)
    out.update({k: float(v) if hasattr(v, "__float__") else v for k,v in metrics.items()})
    history_rows.append(out)
    print("Metrics:", out)

with open(SAFE_OUT / "block_metrics.json", "w") as f:
    json.dump(history_rows, f, indent=2)
print("Saved block-wise checkpoints and metrics.")

BLOCK 01/20 | naca2412 | AoA=0.0 | Re=1e+06
train_multi_airfoil: 1 airfoils, 1 conditions, selected combos=1, running blocks 0..0

BLOCK 1/1
  Airfoil : naca2412
  AoA     : 0.0 deg
  Re      : 1.000e+06
  Steps   : 500
  step   100  loss=1.4165e+00  pde=5.4979e-02  wall=4.2029e-01  lr=7.25e-05  λwall=0.87  λpde=0.27
  step   200  loss=1.9164e+00  pde=5.4963e-02  wall=1.2611e-01  lr=5.25e-05  λwall=0.82  λpde=0.13
  step   300  loss=1.5546e+00  pde=6.5674e-02  wall=6.4910e-02  lr=3.81e-05  λwall=0.45  λpde=0.13
  step   400  loss=1.1239e+00  pde=8.6017e-02  wall=5.9185e-02  lr=2.76e-05  λwall=0.20  λpde=0.13
  step   500  loss=8.1971e-01  pde=1.2413e-01  wall=6.3592e-02  lr=2.00e-05  λwall=0.13  λpde=0.13

Saved checkpoint after this 10-block batch.
Next run will start from block index: 1


NameError: name 'xlim' is not defined

In [31]:
from pathlib import Path
import json
import pandas as pd

from postprocess import compute_lift_drag_on_boundary

RUN_DIR = Path("/content/drive/MyDrive/AeroPINN/safe_exact20")

# load schedule
schedule = json.loads((RUN_DIR / "combo_schedule.json").read_text())

# first finished block
case0 = schedule[0]
print("Case 0:", case0)

airfoil_name = case0["airfoil"]
alpha_deg = float(case0["alpha_deg"])
Re_phys = float(case0["Re_phys"])

# find matching airfoil
airfoil_item = next(a for a in TRAIN_AIRFOILS if a["name"] == airfoil_name)
geometry = airfoil_item["geometry"]

# explicit plotting / flow domain limits
xlim_eval = (-1.0, 2.0)
ylim_eval = (-1.0, 1.0)

metrics = compute_lift_drag_on_boundary(
    model=model,
    geometry=geometry,
    alpha_deg=alpha_deg,
    Re_phys=Re_phys,
    n_surface=240,
    xlim=xlim_eval,
    ylim=ylim_eval,
    device=device,
)

print("\nFIRST BLOCK METRICS")
print("Airfoil :", airfoil_name)
print("AoA     :", alpha_deg)
print("Re      :", Re_phys)
print("CL      :", float(metrics["CL"]))
print("CD      :", float(metrics["CD"]))
print("L/D     :", float(metrics["LD"]))

row = {
    "block": 0,
    "airfoil": airfoil_name,
    "alpha_deg": alpha_deg,
    "Re_phys": Re_phys,
    "CL": float(metrics["CL"]),
    "CD": float(metrics["CD"]),
    "LD": float(metrics["LD"]),
}

df = pd.DataFrame([row])
display(df)
df.to_csv(RUN_DIR / "first_block_metrics.csv", index=False)

Case 0: {'airfoil': 'naca2412', 'alpha_deg': 0.0, 'Re_phys': 1000000.0}

FIRST BLOCK METRICS
Airfoil : naca2412
AoA     : 0.0
Re      : 1000000.0
CL      : 0.007142287876149567
CD      : 0.005817395477606624
L/D     : 1.2277466614815786


,block,airfoil,alpha_deg,Re_phys,CL,CD,LD
0,0,naca2412,0.0,1000000.0,0.007142,0.005817,1.227747


In [ ]:
row = {
    "block": 0,
    "airfoil": airfoil_name,
    "alpha_deg": alpha_deg,
    "Re_phys": Re_phys,
    "CL": float(metrics["CL"]),
    "CD": float(metrics["CD"]),
    "LD": float(metrics["LD"]),
}

df = pd.DataFrame([row])
display(df)

df.to_csv(RUN_DIR / "first_block_metrics.csv", index=False)